In [44]:
import pandas as pd
import os
from functools import reduce

In [22]:
def concatenate(files):
    dfs = []
    for file in files:
        df = pd.read_csv(os.path.join(data_path, file), sep=',')
        dfs.append(df)
    fin_df = pd.concat(dfs, ignore_index=True)
    return fin_df

In [38]:
data_path = '/Users/neil/Desktop/GaTech.nosync/Fall2025/CSE 6740/WiFOP/src/nasa_power/power_ca_2020_2023/interim/'
output_path = '/Users/neil/Desktop/GaTech.nosync/Fall2025/CSE 6740/WiFOP/src/nasa_power/power_ca_2020_2023/interim2/'

comp_dict = {}
for file in os.listdir(data_path):
    factor = file.split('_')[0]
    if factor not in comp_dict:
        comp_dict[factor] = []
    for k in comp_dict.keys():
        if k == factor:
            comp_dict[k].append(file)

In [39]:
comp_dict

{'PRECTOT': ['PRECTOT_202208_tile2.csv',
  'PRECTOT_202006_tile1.csv',
  'PRECTOT_202208_tile1.csv',
  'PRECTOT_202006_tile2.csv',
  'PRECTOT_202307_tile2.csv',
  'PRECTOT_202109_tile1.csv',
  'PRECTOT_202307_tile1.csv',
  'PRECTOT_202109_tile2.csv',
  'PRECTOT_202209_tile2.csv',
  'PRECTOT_202007_tile1.csv',
  'PRECTOT_202209_tile1.csv',
  'PRECTOT_202007_tile2.csv',
  'PRECTOT_202306_tile2.csv',
  'PRECTOT_202108_tile1.csv',
  'PRECTOT_202306_tile1.csv',
  'PRECTOT_202108_tile2.csv',
  'PRECTOT_202009_tile2.csv',
  'PRECTOT_202207_tile1.csv',
  'PRECTOT_202009_tile1.csv',
  'PRECTOT_202207_tile2.csv',
  'PRECTOT_202106_tile2.csv',
  'PRECTOT_202308_tile1.csv',
  'PRECTOT_202106_tile1.csv',
  'PRECTOT_202308_tile2.csv',
  'PRECTOT_202008_tile2.csv',
  'PRECTOT_202206_tile1.csv',
  'PRECTOT_202008_tile1.csv'],
 'WS10M': ['WS10M_202009_tile1.csv',
  'WS10M_202207_tile2.csv',
  'WS10M_202009_tile2.csv',
  'WS10M_202207_tile1.csv',
  'WS10M_202106_tile1.csv',
  'WS10M_202308_tile2.csv',
 

In [40]:
os.makedirs(output_path, exist_ok = True)

for k, v in comp_dict.items():
    fin_df = concatenate(v)
    fin_df_sorted = fin_df.sort_values(by=['YEAR', 'DOY'], ascending=[True, True])
    fin_df_sorted["DATE"] = pd.to_datetime(fin_df_sorted["YEAR"].astype(int), format="%Y") + pd.to_timedelta(fin_df_sorted["DOY"].astype(int) - 1, unit="D")
    fin_df_sorted['DATE'] = fin_df_sorted["DATE"].dt.strftime("%Y-%m-%d")
    fin_df_sorted.drop(['YEAR', 'DOY'], inplace=True, axis=1)
    fin_df_sorted.to_csv(os.path.join(output_path, f'{k}_2020_2023_JUN_SEP.csv'), index=False)

In [42]:
combined_path = '/Users/neil/Desktop/GaTech.nosync/Fall2025/CSE 6740/WiFOP/src/nasa_power/power_ca_2020_2023/final/'

os.makedirs(combined_path, exist_ok=True)

dfs = []
for f in os.listdir(output_path):
    df = pd.read_csv(os.path.join(output_path, f), sep=',')
    dfs.append(df)

In [45]:
merged_df = reduce(lambda left, right: pd.merge(left, right, on=['LAT', 'LON', 'DATE'], how='inner'), dfs)

In [47]:
merged_df

,LAT,LON,T2M,DATE,PS,WS10M,PRECTOTCORR,RH2M
0,32.5,-124.375,15.68,2020-06-01,101.80,5.07,2.30,83.75
1,32.5,-123.750,15.67,2020-06-01,101.77,5.44,2.65,84.78
2,32.5,-123.125,15.71,2020-06-01,101.73,6.01,2.85,85.81
3,32.5,-122.500,15.92,2020-06-01,101.69,6.69,3.61,86.26
4,32.5,-121.875,16.01,2020-06-01,101.65,7.33,3.40,86.62
...,...,...,...,...,...,...,...,...
56215,42.0,-122.500,12.76,2022-09-30,88.19,2.41,0.00,67.37
56216,42.0,-121.875,12.40,2022-09-30,86.25,3.38,0.00,64.39
56217,42.0,-121.250,12.89,2022-09-30,86.06,3.83,0.00,58.09
56218,42.0,-120.625,11.58,2022-09-30,84.11,3.58,0.00,59.59


In [48]:
merged_df2 = reduce(lambda left, right: pd.merge(left, right, on=['LAT', 'LON', 'DATE'], how='outer'), dfs)

In [49]:
merged_df2

,LAT,LON,T2M,DATE,PS,WS10M,PRECTOTCORR,RH2M
0,32.5,-124.375,15.68,2020-06-01,101.80,5.07,2.30,83.75
1,32.5,-124.375,16.26,2020-06-02,101.82,6.97,0.36,82.21
2,32.5,-124.375,16.67,2020-06-03,101.75,6.04,0.31,86.54
3,32.5,-124.375,16.71,2020-06-04,101.27,8.76,0.00,87.47
4,32.5,-124.375,16.59,2020-06-05,101.12,9.59,0.00,84.27
...,...,...,...,...,...,...,...,...
165915,42.0,-114.375,16.50,2023-09-26,NaN,3.73,NaN,30.03
165916,42.0,-114.375,14.48,2023-09-27,NaN,3.61,NaN,41.41
165917,42.0,-114.375,11.16,2023-09-28,NaN,2.88,NaN,49.30
165918,42.0,-114.375,12.20,2023-09-29,NaN,2.72,NaN,38.38


In [51]:
merged_df.to_csv(os.path.join(combined_path, 'NASA_POWER_JUN_SEP_2020_23.csv'), index=False)

In [52]:
merged_df2.to_csv(os.path.join(combined_path, 'NASA_POWER_JUN_SEP_2020_23_FULL.csv'), index=False)

In [53]:
merged_df2.isna().mean().sort_values(ascending=False)

PS             0.158631
T2M            0.157546
PRECTOTCORR    0.157546
RH2M           0.157546
WS10M          0.154894
LAT            0.000000
LON            0.000000
DATE           0.000000
dtype: float64